In [9]:
%load_ext autoreload
%autoreload 2


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
from compare_peps_mwpm_surface_code_full_logical_v6 import (
    compare_peps_mwpm_surface_code_full_logical,
)

In [ ]:


table = compare_peps_mwpm_surface_code_full_logical(
    distance=5,
    p_values=[0.002, 0.005, 0.01, 0.02],
    shots=400,
    rounds=3,
    noisy_round=2,
    target_t=1,
    peps_nkeep=128,
    peps_nsweep=1,
    seed=123,
)

table.pretty_print()

In [ ]:
from compare_peps_mwpm_surface_code_full_logical_v6 import (
    compare_peps_mwpm_surface_code_full_logical,
)

table = compare_peps_mwpm_surface_code_full_logical(
    distance=5,
    p_values=[0.2],
    shots=400,
    rounds=3,
    noisy_round=2,
    target_t=1,
    peps_nkeep=128,
    peps_nsweep=1,
    seed=123,
)

table.pretty_print()

In [ ]:

table = compare_peps_mwpm_surface_code_full_logical(
    distance=5,
    p_values=[0.5],
    shots=1024,
    rounds=3,
    noisy_round=2,
    target_t=1,
    peps_nkeep=64,
    peps_nsweep=1,
    seed=123,
)

table.pretty_print()

In [ ]:
from src.Surface_code_sampler.surface_code_capacity_sampler_full_logical_v9 import (
    inspect_v9_structure,
)

print(inspect_v9_structure(distance=5, rounds=3, memory_basis="x"))
print(inspect_v9_structure(distance=5, rounds=3, memory_basis="z"))

In [37]:
from debug_compare_surface_code import debug_one_point_v6

out = debug_one_point_v6(
    distance=5,
    p=0.08,
    shots=100,
    rounds=3,
    noisy_round=2,
    target_t=1,
    peps_nkeep=128,
    peps_nsweep=1,
    seed=123,
    print_examples=10,
    
)


Sampler truth diagnostics:
  true logical sectors: {'I': 65, 'Z': 17, 'X': 12, 'Y': 6}
  logical_bits bit-1 fraction: [0.23, 0.18]

Batch-X diagnostics (x-memory):
  observable_flips shape: (100, 1)
  observable bit-1 fraction: [0.23]
  detector stats: {'nonzero_fraction': 0.97, 'mean_weight': 6.32, 'max_weight': 16}
  sX stats: {'nonzero_fraction': 0.2222222222222222, 'mean_weight': 0.35444444444444445, 'max_weight': 4}
  sZ stats: {'nonzero_fraction': 0.2411111111111111, 'mean_weight': 0.3477777777777778, 'max_weight': 3}

Batch-Z diagnostics (z-memory):
  observable_flips shape: (100, 1)
  observable bit-1 fraction: [0.18]
  detector stats: {'nonzero_fraction': 0.97, 'mean_weight': 6.32, 'max_weight': 16}
  sX stats: {'nonzero_fraction': 0.2222222222222222, 'mean_weight': 0.35444444444444445, 'max_weight': 4}
  sZ stats: {'nonzero_fraction': 0.2411111111111111, 'mean_weight': 0.3477777777777778, 'max_weight': 3}

Truth consistency checks:
  x-memory observable vs true_bits[:,0] mis

In [12]:
from src.ML_decoder_PEPS.PEPS_Pauli_decoder import * 

In [35]:
from src.Surface_code_sampler.surface_code_capacity_sampler_full_logical_v9 import (
    sample_surface_code_capacity_batch_full_logical_v9,
)

data = sample_surface_code_capacity_batch_full_logical_v9(
    distance=5,
    shots=100,
    p=0.1,
    rounds=3,
    noisy_round=2,
    target_t=1,
    seed=123,
    debug=True,
)

Sampler source file: c:\Users\Minyoung Kim\Desktop\TNDecoder\src\Surface_code_sampler\surface_code_capacity_sampler_full_logical_v9.py
Sampler function: sample_surface_code_capacity_batch_full_logical_v9
DEBUG full-logical sampler v9 (structural)
 repeat_count: 2
 injected repeat iteration: 1
 num inferred round-body data qubits: 41
 logical bit convention: [z_log_from_x_memory, x_log_from_z_memory]
 first 5 true logical bits:
[[0 0]
 [1 0]
 [0 1]
 [1 0]
 [0 0]]
 first 5 observable flips (memory_x -> z_log):
[[0]
 [1]
 [0]
 [1]
 [0]]
 first 5 observable flips (memory_z -> x_log):
[[0]
 [0]
 [1]
 [0]
 [0]]
 average local depolarizing rate: 0.10000000000000009
 zero-syndrome fractions:
 memory_x: 0.46444444444444444
 memory_z: 0.46444444444444444


In [21]:
from src.ML_decoder_PEPS.PEPS_Pauli_decoder import _auto_choose_logical_masks, _induced_syndrome_from_edge_masks

In [24]:
def diagnostic_test_trivial_zero_syndrome(data, p: float, Nkeep: int = 128, Nsweep: int = 1):
    logical_bits = np.asarray(data.logical_bits, dtype=np.uint8)
    idx = None
    for k in range(len(logical_bits)):
        if np.all(logical_bits[k] == 0):
            if np.sum(data.batch.sX[k]) == 0 and np.sum(data.batch.sZ[k]) == 0:
                idx = k
                break

    if idx is None:
        print("No trivial zero-syndrome shot found.")
        return

    nrow, ncol = data.batch.sX[0].shape
    W_h, W_v = depolarizing_weights(nrow, ncol, p=p)

    active_X = np.asarray(data.batch.active_X[0], dtype=np.uint8)
    active_Z = np.asarray(data.batch.active_Z[0], dtype=np.uint8)
    logical_X_masks, logical_Z_masks, info = _auto_choose_logical_masks(active_X, active_Z)

    cosets = pauli_coset_likelihoods_peps(
        sX=data.batch.sX[idx],
        sZ=data.batch.sZ[idx],
        active_X=data.batch.active_X[idx],
        active_Z=data.batch.active_Z[idx],
        W_h=W_h,
        W_v=W_v,
        Nkeep=Nkeep,
        Nsweep=Nsweep,
        logical_X_masks=logical_X_masks,
        logical_Z_masks=logical_Z_masks,
    )
    print("=" * 100)
    print("TRIVIAL ZERO-SYNDROME DIAGNOSTIC")
    print("=" * 100)
    print("shot =", idx)
    print("cosets =", cosets)
def diagnostic_test_cosets_distinguish_representative_shots(
    data,
    p: float,
    Nkeep: int = 128,
    Nsweep: int = 1,
):
    logical_bits = np.asarray(data.logical_bits, dtype=np.uint8)
    sX_all = np.asarray(data.batch.sX, dtype=np.uint8)
    sZ_all = np.asarray(data.batch.sZ, dtype=np.uint8)

    # find one low-syndrome X and one low-syndrome Z shot
    rep_X = None
    rep_Z = None
    best_wX = None
    best_wZ = None

    for k in range(len(logical_bits)):
        bits = tuple(map(int, logical_bits[k]))
        w = int(np.sum(sX_all[k]) + np.sum(sZ_all[k]))
        if bits == (0, 1):
            if rep_X is None or w < best_wX:
                rep_X = k
                best_wX = w
        if bits == (1, 0):
            if rep_Z is None or w < best_wZ:
                rep_Z = k
                best_wZ = w

    if rep_X is None or rep_Z is None:
        raise RuntimeError("Could not find representative X/Z shots.")

    nrow, ncol = sX_all[0].shape
    W_h, W_v = depolarizing_weights(nrow, ncol, p=p)

    active_X = np.asarray(data.batch.active_X[0], dtype=np.uint8)
    active_Z = np.asarray(data.batch.active_Z[0], dtype=np.uint8)
    logical_X_masks, logical_Z_masks, info = _auto_choose_logical_masks(active_X, active_Z)

    print("=" * 100)
    print("REPRESENTATIVE SHOT DIAGNOSTIC")
    print("=" * 100)
    print("rep_X =", rep_X, "true bits =", logical_bits[rep_X].tolist(),
          "syndrome weight =", int(np.sum(sX_all[rep_X]) + np.sum(sZ_all[rep_X])))
    print("rep_Z =", rep_Z, "true bits =", logical_bits[rep_Z].tolist(),
          "syndrome weight =", int(np.sum(sX_all[rep_Z]) + np.sum(sZ_all[rep_Z])))

    cos_X = pauli_coset_likelihoods_peps(
        sX=data.batch.sX[rep_X],
        sZ=data.batch.sZ[rep_X],
        active_X=data.batch.active_X[rep_X],
        active_Z=data.batch.active_Z[rep_X],
        W_h=W_h,
        W_v=W_v,
        Nkeep=Nkeep,
        Nsweep=Nsweep,
        logical_X_masks=logical_X_masks,
        logical_Z_masks=logical_Z_masks,
    )
    cos_Z = pauli_coset_likelihoods_peps(
        sX=data.batch.sX[rep_Z],
        sZ=data.batch.sZ[rep_Z],
        active_X=data.batch.active_X[rep_Z],
        active_Z=data.batch.active_Z[rep_Z],
        W_h=W_h,
        W_v=W_v,
        Nkeep=Nkeep,
        Nsweep=Nsweep,
        logical_X_masks=logical_X_masks,
        logical_Z_masks=logical_Z_masks,
    )

    print("\nCosets for representative X shot:")
    print(cos_X)
    print("\nCosets for representative Z shot:")
    print(cos_Z)

    vals_X = np.array([cos_X[(0,0)], cos_X[(1,0)], cos_X[(0,1)], cos_X[(1,1)]], dtype=float)
    vals_Z = np.array([cos_Z[(0,0)], cos_Z[(1,0)], cos_Z[(0,1)], cos_Z[(1,1)]], dtype=float)

    print("\nAre X/Z representative cosets identical?")
    print(np.allclose(vals_X, vals_Z, atol=1e-15, rtol=1e-12))
def diagnostic_test_logical_masks_from_batch(batch):
    active_X = np.asarray(batch.active_X[0], dtype=np.uint8)
    active_Z = np.asarray(batch.active_Z[0], dtype=np.uint8)

    logical_X_masks, logical_Z_masks, info = _auto_choose_logical_masks(active_X, active_Z)

    sxX, szX = _induced_syndrome_from_edge_masks(active_X, active_Z, *logical_X_masks)
    sxZ, szZ = _induced_syndrome_from_edge_masks(active_X, active_Z, *logical_Z_masks)

    print("=" * 100)
    print("LOGICAL MASK DIAGNOSTIC")
    print("=" * 100)
    print("chosen X mask meta:", info["meta_x"])
    print("chosen Z mask meta:", info["meta_z"])
    print("weight_x =", info["weight_x"])
    print("weight_z =", info["weight_z"])
    print("symplectic overlap =", info["overlap"])
    print("num zero-X candidates =", info["num_zero_x"])
    print("num zero-Z candidates =", info["num_zero_z"])

    print("\nInduced syndrome of logical X mask:")
    print("sum sX =", int(np.sum(sxX)), "sum sZ =", int(np.sum(szX)))
    print("sX nonzeros:", np.argwhere(sxX))
    print("sZ nonzeros:", np.argwhere(szX))

    print("\nInduced syndrome of logical Z mask:")
    print("sum sX =", int(np.sum(sxZ)), "sum sZ =", int(np.sum(szZ)))
    print("sX nonzeros:", np.argwhere(sxZ))
    print("sZ nonzeros:", np.argwhere(szZ))

In [25]:
diagnostic_test_logical_masks_from_batch(data.batch)
diagnostic_test_cosets_distinguish_representative_shots(data, p=0.01, Nkeep=128, Nsweep=1)
diagnostic_test_trivial_zero_syndrome(data, p=0.01, Nkeep=128, Nsweep=1)

LOGICAL MASK DIAGNOSTIC
chosen X mask meta: {'kind': 'X', 'orientation': 'h', 'row': 0, 'parity': 0}
chosen Z mask meta: {'kind': 'Z', 'orientation': 'h', 'row': 0, 'parity': 0}
weight_x = 5
weight_z = 5
symplectic overlap = 1
num zero-X candidates = 22
num zero-Z candidates = 22

Induced syndrome of logical X mask:
sum sX = 0 sum sZ = 0
sX nonzeros: []
sZ nonzeros: []

Induced syndrome of logical Z mask:
sum sX = 0 sum sZ = 0
sX nonzeros: []
sZ nonzeros: []
REPRESENTATIVE SHOT DIAGNOSTIC
rep_X = 69 true bits = [0, 1] syndrome weight = 4
rep_Z = 99 true bits = [1, 0] syndrome weight = 1

Cosets for representative X shot:
{(0, 0): 1.7988687328150938e-07, (1, 0): 1.7988687328150938e-07, (0, 1): 1.7988687328150938e-07, (1, 1): 1.7988687328150938e-07}

Cosets for representative Z shot:
{(0, 0): 0.009304042590733632, (1, 0): 0.009304042590733632, (0, 1): 0.009304042590733632, (1, 1): 0.009304042590733641}

Are X/Z representative cosets identical?
False
TRIVIAL ZERO-SYNDROME DIAGNOSTIC
shot 